# Introducción

```{index} SQL
```

En este apartado vamos a ver como iteractuamos con bases de datos ## Enseñar a los agentes a recuperar y actualizar datos estructurados.

Los agentes obtienen un nivel de utilidad completamente nuevo al poder acceder y modificar datos privados y estructurados, como registros internos, sistemas de tickets o bases de datos de clientes.

A diferencia de los datos web públicos o las API, este tipo de información reside dentro de una organización y suele almacenarse en formatos estructurados que requieren una interacción precisa.

Si se le pregunta a un agente “¿Cuál es el estado del ticket n.° 9265?”, no puede buscarlo en la web, o al menos ahí será muy difícil encuentre la información solicitada. La respuesta probablemente se encuentre en una base de datos relacional o un CRM. De
igual manera, si el agente completa una tarea, debería actualizar el sistema correspondiente; no solo confirmar el éxito en el chat, sino escribir una nueva entrada en la base de datos. Los agentes que interactúan con datos estructurados pueden realizar trabajo real, no solo generar texto.

## Comprensión de las bases de datos

Existen muchos tipos de bases de datos que según la estructura de esos datos, y a grandes rasgos se pueden casificar como:

* **Las bases de datos relacionales** (por ejemplo, PostgreSQL, MySQL) almacenan datos en tablas con esquemas definidos, lo que resulta perfecto para interacciones de agentes que requieren consultas estructuradas.

* **Las bases de datos NoSQL** (por ejemplo, MongoDB) ofrecen flexibilidad, pero se utilizan con menos frecuencia para la integración directa de agentes.

* **Las bases de datos vectoriales** (por ejemplo, Chroma, Pinecone) almacenan incrustaciones para la búsqueda semántica, lo que permite la recuperación basada en similitud, una parte clave de muchos flujos de trabajo RAG (Recuperación-Generación Aumentada).

Pero en este apartado vamos a poner el enfoque en las bases de datos relacional, en las que se accede a la información mediante el lenguaje SQL

## text2SQL: Traduciendo el lenguaje a consultas
```{index} text2SQL
```

Los agentes pueden traducir lenguaje natural a SQL, una práctica conocida como `text2SQL`. Por ejemplo:

*Entrada*: “¿Cuántos usuarios se registraron esta semana?”

**Producción*: SELECT COUNT(*) FROM users WHERE signup_date >= ‘2025-07-01’

Esto requiere que el agente:

* Comprender los esquemas de bases de datos

* Manejar filtros y rangos de fechas

*  Evite consultas peligrosas (por ejemplo, accidentales DELETE)

Para evitar errores críticos, los agentes del mundo real deben estar protegidos con medidas de seguridad, incluida la validación de consultas y las aprobaciones humanas.

Como resumen de esta introducción, podemos decir que:

* La recuperación de datos estructurados permite a los agentes realizar tareas significativas dentro de sistemas privados.

*  Los agentes pueden leer y escribir datos utilizando SQL con medidas de seguridad implementadas.

*  Las bases de datos vectoriales permiten la búsqueda semántica, pero no son la única forma de realizar RAG.

* La combinación de fuentes de datos relacionales y vectoriales proporciona a los agentes capacidades de razonamiento más profundas y flexibles.

Los agentes más eficaces son aquellos que pueden integrarse con sistemas estructurados, tomar decisiones informadas y producir resultados que importan en contextos del mundo real.

Comenzamos este apartado viendo determinados aspectos de la base de datos sqlite para posteriormente ir avanzando en nivel de conocimientos hasta llegar a usar `sqlalchemy` que supone un nivel avanzado en el tratamiento de las bases de datos con python.

## sqlite
```{index} sqlite
```
`SQLite` es un sistema de gestión de bases de datos relacional muy ligero y ampliamente utilizado. A diferencia de otros sistemas como MySQL o PostgreSQL, SQLite no funciona como un servidor independiente, sino que está integrado directamente dentro de la aplicación que lo usa.

Esto significa que no requiere instalación ni configuración compleja: toda la base de datos se guarda en un único archivo en el sistema, lo que lo hace ideal para aplicaciones móviles, programas de escritorio o proyectos pequeños.

SQLite utiliza el lenguaje estándar SQL para realizar operaciones como crear tablas, insertar datos, consultarlos o modificarlos. A pesar de su simplicidad, es muy potente y cumple con gran parte del estándar SQL, incluyendo transacciones seguras.

En resumen, SQLite es una opción práctica cuando necesitas una base de datos rápida, sencilla y sin complicaciones de administración.





In [ ]:
# Importamos sqlite (antes se debetener instalado db-sqlite3)
import sqlite3

Con esta esta herramienta podemos crear una base de datos en local, que por ejemplo se llame `employe.db`, de la siguiente manera:

In [ ]:
connection = sqlite3.connect("employee.db")

Al ejecutar la instrucción anterior, podemos abservar que se ha creado un fichero denominado `employe.db` en la misma carpeta que donde está este fichero jupyter. Veamos a continuación la información que obtenemos con este objeto:

In [ ]:
connection

Hasta esto momento lo que tenemos es una base de datos, pero sin tablas, a continuación creamos tres tablas que serán la base de futuros trabajos expositivos sobre esta materia.

In [ ]:
table_creation_query="""
CREATE TABLE IF NOT EXISTS employees (
    emp_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    hire_date TEXT NOT NULL,
    salary REAL NOT NULL
);
"""

table_creation_query2="""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    phone TEXT
);
"""

table_creation_query3="""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);

"""
# Creamos un objeto cursor
cursor=connection.cursor()
# generamos realmente la primera tabla
cursor.execute(table_creation_query)

# generamos realmente la segunda tabla
cursor.execute(table_creation_query2)

# generamos realmente la tercera tabla
cursor.execute(table_creation_query3)

Los esquemas UML generado con estas instrucciones son los siguientes:

![](../img/esquemaUML.jpg)

A continuación procedemos a llenar las tablas con valores, para ello, lo primero que hacemos es crear las plantillas correspondientes:

In [ ]:
#plantilla para employes
insert_query = """
INSERT INTO employees (emp_id, first_name, last_name, email, hire_date, salary)
VALUES (?, ?, ?, ?, ?, ?);
"""

# plantilla para customers
insert_query_customers = """
INSERT INTO customers (customer_id, first_name, last_name, email, phone)
VALUES (?, ?, ?, ?, ?);
"""

# plantilla para orders
insert_query_orders = """
INSERT INTO orders (order_id, customer_id, order_date, amount)
VALUES (?, ?, ?, ?);
"""

Ahora definimos realmente los datos que vamos a incluir

In [ ]:
employee_data = [
    (1, "Sunny", "Savita", "sunny.sv@abc.com", "2023-06-01", 50000.00),
    (2, "Arhun", "Meheta", "arhun.m@gmail.com", "2022-04-15", 60000.00),
    (3, "Alice", "Johnson", "alice.johnson@jpg.com", "2021-09-30", 55000.00),
    (4, "Bob", "Brown", "bob.brown@uio.com", "2020-01-20", 45000.00),
    ]

customers_data = [
    (1, "John", "Doe", "john.doe@example.com", "1234567890"),
    (2, "Jane", "Smith", "jane.smith@example.com", "9876543210"),
    (3, "Emily", "Davis", "emily.davis@example.com", "4567891230"),
    (4, "Michael", "Brown", "michael.brown@example.com", "7894561230"),
]

orders_data = [
    (1, 1, "2023-12-01", 250.75),
    (2, 2, "2023-11-20", 150.50),
    (3, 3, "2023-11-25", 300.00),
    (4, 4, "2023-12-02", 450.00),
]

Ahora incorporamos realmente los datos:

In [ ]:
cursor.executemany(insert_query,employee_data)
connection.commit()

In [ ]:

cursor.executemany(insert_query_customers,customers_data)

cursor.executemany(insert_query_orders,orders_data)

connection.commit()

In [17]:
# Veamos el contenido de los registros
cursor.execute("select * from employees;")
for row in cursor.fetchall():
    print(row)

(1, 'Sunny', 'Savita', 'sunny.sv@abc.com', '2023-06-01', 50000.0)
(2, 'Arhun', 'Meheta', 'arhun.m@gmail.com', '2022-04-15', 60000.0)
(3, 'Alice', 'Johnson', 'alice.johnson@jpg.com', '2021-09-30', 55000.0)
(4, 'Bob', 'Brown', 'bob.brown@uio.com', '2020-01-20', 45000.0)


In [18]:
cursor.execute("select first_name from employees where salary > 50000.0;")
cursor.fetchall()

[('Arhun',), ('Alice',)]

In [19]:
cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table';")
cursor.fetchall()

[('employees',), ('customers',), ('sqlite_sequence',), ('orders',)]

## LLM con Groq
```{index} Groq
```

Hasta estos momentos, se ha utilizado una gestión de base de datos de modo tradicional. En lo que sigue, introducimos el modelo LLM con Groq, pues ademá de ser generoso con la parte gratuita, nos va a permitir desarrollar los conceptos fundamentales perseguidoes en este apartado. 

Comenzamos leyendo la api key que nos proporciona esta LLM (se puede obtener, previa identificación en https://console.groq.com/keys)

In [27]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"]= GROQ_API_KEY

from langchain_groq import ChatGroq
llm=ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

In [29]:
# Probamos el modelo anterior
llm.invoke("Buenos días, como estás?").content

'**Buenos días**\n\nEstoy muy bien, gracias por preguntar. Me alegra poder conversar contigo. ¿En qué puedo ayudarte hoy? ¿Tienes algún tema en particular que te gustaría discutir o alguna pregunta que te esté molestando? Estoy aquí para escucharte y ayudarte en lo que pueda. ¿Quieres hablar sobre algo en específico o simplemente charlar un rato?'

```{index} SQLDatabase
```
Preparamos a continuación todo lo necesario para utilizar modelos y herramientas que nos proporciona LangChain. Comenzamos con el usos de una herramienta que nos proporciona la comunidad de LangChain, como es SQLDatabase.

In [ ]:
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///employee.db")

print("Dialect:", db.dialect)
print("Tablas en la base de datos:", db.get_usable_table_names())

Dialect: sqlite
Tablas utilizadas: ['customers', 'employees', 'orders']


In [33]:
query_result = db.run("SELECT * FROM employees ;")
print("Registros que hay en la BD employees: \n", query_result)

Registros que hay en la BD employees: 
 [(1, 'Sunny', 'Savita', 'sunny.sv@abc.com', '2023-06-01', 50000.0), (2, 'Arhun', 'Meheta', 'arhun.m@gmail.com', '2022-04-15', 60000.0), (3, 'Alice', 'Johnson', 'alice.johnson@jpg.com', '2021-09-30', 55000.0), (4, 'Bob', 'Brown', 'bob.brown@uio.com', '2020-01-20', 45000.0)]


```{index} SQLDatabaseToolkit
```

Utilizar este tipo de herramientas nos proporciona la ventaja de que facilitan una serie de herramientas (Tools) que nos facilitan mucho el trabajo. En este caso concreto vamos a ver las herramientas que la comunidad de LangChain nos ofrece para el trabajo con bases de datos (Ver el siguiente enlace: https://reference.langchain.com/python/langchain-community/agent_toolkits/sql/toolkit/SQLDatabaseToolkit).

In [35]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit=SQLDatabaseToolkit(db=db,llm=llm)
tools=toolkit.get_tools()
tools

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000265DE4BFFE0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000265DE4BFFE0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000265DE4BFFE0>),
 QuerySQLCheckerTool(description='Use this tool to 

In [36]:
for tool in tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [37]:
list_tables_tool = next((tool for tool in tools if tool.name == "sql_db_list_tables"), None)
list_tables_tool

ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000265DE4BFFE0>)

In [38]:
get_schema_tool = next((tool for tool in tools if tool.name == "sql_db_schema"), None)
get_schema_tool

InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000265DE4BFFE0>)

Con los objetos creados de esta manera, podemos obtener, por ejemplo,  una lista de tablas existentes en la base de datos o el esquema de una determinada tabla de la base de datos

In [39]:
print(list_tables_tool.invoke(""))

customers, employees, orders


In [40]:
print(get_schema_tool.invoke("employees"))


CREATE TABLE employees (
	emp_id INTEGER, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	hire_date TEXT NOT NULL, 
	salary REAL NOT NULL, 
	PRIMARY KEY (emp_id), 
	UNIQUE (email)
)

/*
3 rows from employees table:
emp_id	first_name	last_name	email	hire_date	salary
1	Sunny	Savita	sunny.sv@abc.com	2023-06-01	50000.0
2	Arhun	Meheta	arhun.m@gmail.com	2022-04-15	60000.0
3	Alice	Johnson	alice.johnson@jpg.com	2021-09-30	55000.0
*/


El esquema que seguiremos en esta exposición es el siguiente:

![](../img/esquemaTool.png)

Con todo este potencial que nos ofrecen las Tools mostradas anteriormente, procedemos a crear una Tool personalizada que nos puede servir bien para desrrollos posteriores

In [41]:
from langchain_core.tools import tool
@tool
def db_query_tool(query: str) -> str:
    """
    Execute a SQL query against the database and return the result.
    If the query is invalid or returns no result, an error message will be returned.
    In case of an error, the user is advised to rewrite the query and try again.
    """
    result = db.run_no_throw(query)
    if not result:
        return "Error: Query fallado. Escribe la query de nuevo e inténtalo de nuevo."
    return result

Probemos la Tool o herramienta anterior para ver como funciona

In [42]:
print(db_query_tool.invoke("SELECT * FROM Employees LIMIT 5;"))

[(1, 'Sunny', 'Savita', 'sunny.sv@abc.com', '2023-06-01', 50000.0), (2, 'Arhun', 'Meheta', 'arhun.m@gmail.com', '2022-04-15', 60000.0), (3, 'Alice', 'Johnson', 'alice.johnson@jpg.com', '2021-09-30', 55000.0), (4, 'Bob', 'Brown', 'bob.brown@uio.com', '2020-01-20', 45000.0)]


Creamos a continuación una clase que nos va a servir para obtener una respuesta estructurada, es decir una respuesta con un formato que es el que nosotros queremos ([ver este apartado](templates)).

In [46]:
from pydantic import BaseModel, Field

class SubmitFinalAnswer(BaseModel):
    """Submit the final answer to the user based on the query results."""
    final_answer: str = Field(..., description="The final answer to the user")

También hay que definir el [estado del agente](estado). Lo hacemos a continuación 

In [54]:
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

https://www.youtube.com/watch?v=gcRSQblTwPw&t=724s

https://github.com/sunnysavita10/langgraph-end-to-end/blob/main/agent_based_rag/sql_agent_with_langgraph.ipynb